# 01 — Reprodução: detecção de mixers na Ethereum

Notebook didático de reprodução do projeto inspirado no artigo *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (SBSeg 2025).

**Escopo desta versão:** utilizar datasets processados e auditados, verificar integridade, validar e treinar Random Forest, e avaliar o dia 29/10/2020. **Não reprocessa dumps brutos** e não deve ser apresentado como validação cega. Código versionado no [GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection/tree/main/src/ethmix); esta cópia é para execução no Colab.

**Unidade:** janela contrato-destinatário/dia UTC. Um endereço pode produzir várias janelas. Conjunto de treino legado tem 23 positivas de **um único pool** Tornado Cash 10 ETH em março/2025 e 100 negativas selecionadas por inspeção; isso limita a generalização.
**Dados oficiais:** Projeto_Mixers_Ethereum no seu Drive (não o atalho do Bruno).

**Acesso aos dados:** pelo ID de cada arquivo do seu Drive; não exige que a pasta apareça dentro de `MyDrive`.


## 1 — Conceitos e variáveis

| Feature | O que mede |
|---|---|
| `value_entropy` | Diversidade dos valores transferidos |
| `fee_entropy` | Diversidade das taxas |
| `internal_value_entropy` | Diversidade dos valores internos |
| `call_count_mean` | Média das chamadas da transação |

As entropias são Shannon base 2, depois de discretização por arredondamento (4 casas valores, 6 taxas). Parâmetros de engenharia deste projeto não são automaticamente parâmetros explícitos do artigo.

## 2 — Carregar o código diretamente do **seu** Google Drive

A pasta `Projeto_Mixers_Ethereum` está no Drive, mas pode não aparecer na montagem `/content/drive/MyDrive` (por exemplo, se estiver em um local compartilhado). **Não precisamos depender desse caminho.**

Esta célula solicita o login **Google** do Colab (não é token GitHub), acessa **somente os três módulos Python já copiados para o seu projeto**, pelos respectivos IDs do Drive, e confere SHA-256 para garantir que o código corresponde à versão registrada no GitHub.

Os módulos são baixados para a pasta temporária `/content/ethmix_reproducao/src/ethmix/`. Não alteramos dados da pasta compartilhada do Bruno.

In [ ]:
from pathlib import Path
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, json, sys, hashlib

# Autoriza somente a conta Google no Colab. Não precisa montar o Meu Drive.
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_api = build('drive', 'v3', credentials=creds, cache_discovery=False)

REPO = 'BorgesGusta/ethereum-mixer-detection'
CODIGO_REFERENCIA = 'src/ethmix (GitHub, hash SHA-256 dos módulos)'
BASE = Path('/content/ethmix_reproducao')  # cache de execução, NÃO o Drive compartilhado

# IDs dos módulos copiados para o seu Projeto_Mixers_Ethereum/src/ethmix.
MODULOS_DRIVE = {
    '__init__.py': '1DnOyPTb_N_GztcgZ5BDp28vp5utfwBph',
    'features.py': '1rxO8fSvW5GABo7YmYVJtcuTTmMvnBQzL',
    'models.py': '10x_kaEy6pHRRYULoud5lFvFZLqGYy6OT',
}
EXPECTED_MODULE_SHA256 = {
    '__init__.py': 'dccb63bbe5fbde892d3059dab7e3efc03f2e0e169d93afa67e8d983e996cd1ae',
    'features.py': 'b75d65bd2326169e6fc52f86f5030337d6e6813df607bab0a9c3cb20922fd0b5',
    'models.py': '7f92d88cafcfc7c939f8c84e4fcfb34a0aa98c0890a86333c294b23f1508d6d9',
}

def obter_arquivo_drive(file_id, destino):
    """Baixa arquivo binário do próprio Drive por ID, sem precisar de um caminho montado."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with destino.open('wb') as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=4 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return destino

for nome, file_id in MODULOS_DRIVE.items():
    arquivo = obter_arquivo_drive(file_id, BASE / 'src' / 'ethmix' / nome)
    obtido = hashlib.sha256(arquivo.read_bytes()).hexdigest()
    if obtido != EXPECTED_MODULE_SHA256[nome]:
        raise ValueError(f'SHA-256 diferente para {nome}: {obtido}. Verifique a sincronização GitHub/Drive.')

sys.path.insert(0, str(BASE / 'src'))
# Evita módulos antigos da memória se uma versão anterior do notebook já foi executada.
for nome in ('ethmix.models', 'ethmix.features', 'ethmix'):
    sys.modules.pop(nome, None)
from ethmix.features import FEATURES, DECIMAIS, preparar, calcular_features
from ethmix.models import novo_rf, validar_cv, reamostrar
print('Código carregado do seu Drive (por ID), sem GitHub Token.')
print('Integridade SHA-256 dos três módulos: verificada.')

## 3 — Baixar os dados processados **da sua pasta** e conferir a integridade

Os dois arquivos Parquet já foram copiados para `Projeto_Mixers_Ethereum/data/processed/`. O Colab os obtém **pelos IDs das cópias no seu Drive**, independentemente de onde a pasta aparece na interface. Conferimos seus SHA-256 com o inventário auditado.

Os downloads são temporários. **Não altera nem sobrescreve os datasets originais.**

In [ ]:
import pandas as pd, numpy as np
from datetime import datetime, timezone

# Arquivos COPIADOS para seu próprio projeto, não arquivos da pasta do Bruno.
DADOS_DRIVE = {
    'treino': '1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S',
    'teste2020': '1g22eCccxC4v_S2RmxPhFZyzfF1ARuzPk',
}
INPUTS = {
    'treino': BASE / 'data' / 'processed' / 'features_treino.parquet',
    'teste2020': BASE / 'data' / 'processed' / 'features_teste_20201029.parquet',
}
HASHES_ESPERADOS = {
    'treino': '0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32',
    'teste2020': '6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51',
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloco in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

for chave, file_id in DADOS_DRIVE.items():
    arquivo = obter_arquivo_drive(file_id, INPUTS[chave])
    obtido = sha256(arquivo)
    print(chave, arquivo.name, 'SHA-256:', obtido)
    if obtido != HASHES_ESPERADOS[chave]:
        raise ValueError(f'Arquivo {chave} divergente. Não continuar sem auditoria.')
print('Dados do seu Drive: integridade SHA-256 verificada.')

## 4 — Inspecionar as amostras

**Treino (legado):** 123 janelas, 23 positivas e 100 negativas, com positivos de um único contrato. A validação cruzada estratificada por linha não comprova generalização entre contratos.

**Teste (29/10/2020):** apenas contratos com pelo menos 10 transações entram na classificação. Os demais endereços não são negativos verificados.

In [ ]:
treino=pd.read_parquet(INPUTS["treino"])
teste=pd.read_parquet(INPUTS["teste2020"])
MIN_TX=10
SEED=42
assert set(FEATURES).issubset(treino.columns) and set(FEATURES).issubset(teste.columns)
assert {"label","n_tx"}.issubset(treino.columns)
assert {"recipient","n_tx","tornado"}.issubset(teste.columns)
assert set(treino.label.dropna().unique())=={0,1}
treino_e=treino.loc[treino.n_tx>=MIN_TX].copy()
teste_e=teste.loc[teste.n_tx>=MIN_TX].copy()
print("Treino completo:",len(treino),"classes:",treino.label.value_counts().to_dict())
print("Treino elegível:",len(treino_e),"classes:",treino_e.label.value_counts().to_dict())
print("Teste 2020:",len(teste),"janelas; elegíveis:",len(teste_e))
print("Pools de referência:",int(teste_e.tornado.notna().sum()))
display(treino_e.groupby("label")[list(FEATURES)].median().round(3))

## 5 — Validar e treinar o classificador

Random Forest com 100 árvores, profundidade máxima 5, pesos balanceados e semente 42. Oversampling apenas dentro dos folds de treino. As métricas são **internas e exploratórias**, não evidência de teste independente.

In [ ]:
cv=validar_cv(treino_e,seed=SEED,n_splits=5)
display(cv.round(3))
display(cv[["precision","recall","f1"]].agg(["mean","std"]).round(3))
Xt,yt=reamostrar(treino_e[list(FEATURES)],treino_e.label.astype(int),SEED)
modelo=novo_rf(SEED).fit(Xt,yt)
print("RF treinada",modelo.n_estimators,"árvores; max_depth",modelo.max_depth)

## 6 — Aplicar ao dia histórico de 2020

Testamos limiares 0,50 e 0,85 **apenas para caracterização exploratória**. Os demais alertas são endereços sem rótulos confirmados; não podem ser chamados automaticamente de falsos positivos.

A referência de 63 endereços no artigo é um ponto de comparação, **não** um alvo de otimização.

In [ ]:
prob=modelo.predict_proba(teste_e[list(FEATURES)])[:,1]
rows=[]
for limiar in [0.50,0.85]:
    mask=prob>limiar
    recuperados=int(teste_e.loc[mask,"tornado"].notna().sum())
    rows.append({"limiar":limiar,"enderecos_sinalizados":int(mask.sum()),"pools_referencia_recuperados":recuperados,"outros_sinalizados_sem_rotulo_confirmado":int(mask.sum())-recuperados})
resumo=pd.DataFrame(rows)
display(resumo)
print("Confrontar estes números com os relatórios originais antes de afirmar paridade")

## 7 — Registrar a execução **no seu Drive**, sem sobrescrever o legado

Os relatórios são primeiro preparados em `/content/ethmix_reproducao/results/`, depois **enviados por API do Google Drive** para a pasta `Projeto_Mixers_Ethereum/results/`, usando seu ID. Isso funciona mesmo que o projeto não esteja montado em `MyDrive`.

Cada execução cria uma pasta exclusiva em `results/reproducao_consolidada/AAAA...Z/`, com CSVs e manifesto. Os modelos e dados originais não são sobrescritos.

**Atenção:** os resultados ainda precisam ser confrontados com o legado; não se trata de validação cega.

In [ ]:
ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = BASE / 'results' / 'reproducao_consolidada' / ts
OUT.mkdir(parents=True, exist_ok=False)
cv.to_csv(OUT / 'cv_folds.csv', index=False)
resumo.to_csv(OUT / 'resumo_2020.csv', index=False)
manifesto = {
    'experimento': 'reproducao_consolidada_01',
    'data_utc': ts,
    'repo': REPO,
    'codigo_ref': CODIGO_REFERENCIA,
    'codigo_sha256': EXPECTED_MODULE_SHA256,
    'arquivos_drive_ids': {'modulos': MODULOS_DRIVE, 'dados': DADOS_DRIVE},
    'dados_sha256': HASHES_ESPERADOS,
    'min_tx': MIN_TX, 'seed': SEED,
    'rf': {'n_estimators': 100, 'max_depth': 5, 'class_weight': 'balanced'},
    'limiares_exploratorios': [0.50, 0.85],
    'ressalvas': ['positivos de um unico contrato',
                 'enderecos nao rotulados nao sao negativos',
                 'teste 2020 previamente explorado']
}
(OUT / 'manifesto.json').write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')

# IDs das pastas que pertencem ao seu Projeto_Mixers_Ethereum.
PASTA_RESULTADOS_ID = '1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'

def obter_ou_criar_pasta(nome, parent_id):
    seguro = nome.replace("'", "\'")
    query = (f"name = '{seguro}' and '{parent_id}' in parents and "
             "mimeType = 'application/vnd.google-apps.folder' and trashed = false")
    encontrados = drive_api.files().list(
        q=query, fields='files(id,name),nextPageToken', supportsAllDrives=True,
        includeItemsFromAllDrives=True, pageSize=100).execute().get('files', [])
    if encontrados:
        return encontrados[0]['id']
    nova = drive_api.files().create(body={
        'name': nome, 'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]}, fields='id').execute()
    return nova['id']

pasta_experimento = obter_ou_criar_pasta('reproducao_consolidada', PASTA_RESULTADOS_ID)
pasta_execucao = obter_ou_criar_pasta(ts, pasta_experimento)
for nome in ('cv_folds.csv', 'resumo_2020.csv', 'manifesto.json'):
    arquivo = OUT / nome
    tipo = 'application/json' if nome.endswith('.json') else 'text/csv'
    media = MediaFileUpload(str(arquivo), mimetype=tipo, resumable=True)
    drive_api.files().create(body={'name': nome, 'parents': [pasta_execucao]},
                             media_body=media, fields='id').execute()
print('Resultados registrados no seu Drive:')
print('https://drive.google.com/drive/folders/' + pasta_execucao)

## 8 — Por que o artigo sinalizou 63 endereços e nossa reprodução 9?

**Pergunta da pesquisa:** ambos identificaram os três pools Tornado Cash em 29/10/2020, mas o artigo relata **63 endereços sinalizados** e nosso teste, no limiar **0,50**, relata **9**. Vamos investigar sem alterar o modelo nem tentar forçar o resultado a 63.

**O que faremos:** contar as janelas e endereços elegíveis; examinar a distribuição das probabilidades de *mixer*; mostrar os primeiros endereços por pontuação e o comportamento dos alertas ao variar **apenas para diagnóstico** o limiar. Vamos salvar os dados que faltavam no experimento 01: **identidades e scores de todos os endereços elegíveis**.

**O que os gráficos NÃO demonstram:** não indicam qual escolha metodológica dos autores causou a diferença. Não conhecemos a lista dos 63 alertas deles e não devemos confundir alteração do limiar com reprodução fiel do artigo.

**Pré-requisito:** execute as etapas 2–6 deste notebook na mesma sessão, para deixar `teste_e`, `modelo`, `FEATURES` e `prob` disponíveis.


In [ ]:
#@title 5. Investigar população, probabilidades e limites — sem retreinar
import matplotlib.pyplot as plt
from IPython.display import display

predicoes = teste_e[['recipient', 'n_tx', 'tornado']].copy().reset_index(drop=True)
predicoes['prob_mixer'] = modelo.predict_proba(teste_e[list(FEATURES)])[:, 1]
predicoes = predicoes.sort_values('prob_mixer', ascending=False).reset_index(drop=True)
predicoes['sinalizado_050'] = predicoes['prob_mixer'] > 0.50

contagens = pd.DataFrame([{
    'janelas_processadas': len(teste),
    'janelas_elegiveis_n_tx_10': len(teste_e),
    'enderecos_distintos_elegiveis': teste_e['recipient'].nunique(),
    'alertas_no_limiar_050': int(predicoes.sinalizado_050.sum()),
    'pools_tornado_conhecidos_recuperados': int(predicoes.loc[predicoes.sinalizado_050, 'tornado'].notna().sum()),
    'alertas_publicados_pelos_autores': 63,
}])
display(contagens.T.rename(columns={0: 'Quantidade'}))
display(predicoes.head(15).rename(columns={'prob_mixer':'P(mixer)'}))

limiares = np.arange(0.05, 1.00, 0.05)
curva = pd.DataFrame({'limiar': limiares,
                      'enderecos_sinalizados': [int((predicoes.prob_mixer > t).sum()) for t in limiares],
                      'tornado_recuperados': [int(predicoes.loc[predicoes.prob_mixer > t, 'tornado'].notna().sum()) for t in limiares]})

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
axs[0].hist(predicoes.prob_mixer, bins=np.linspace(0, 1, 21))
axs[0].axvline(.5, linestyle='--', label='Limiar original 0,50')
axs[0].set(xlabel='Probabilidade estimada de mixer', ylabel='Janelas elegíveis',
           title='A · Distribuição dos scores')
axs[0].legend(fontsize=8)
axs[1].plot(curva.limiar, curva.enderecos_sinalizados, marker='o', markersize=3,
            label='Alertas na nossa reprodução')
axs[1].axhline(63, linestyle=':', label='63 alertas relatados no artigo')
axs[1].axvline(.5, linestyle='--', label='Limiar original 0,50')
axs[1].set(xlabel='Limiar exploratório (não ajustado)', ylabel='Quantidade de alertas',
           title='B · Sensibilidade ao limiar')
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
print('A curva ajuda a entender o score. Ajustar o limiar para chegar a 63 NÃO é reprodução fiel.')


### Como interpretar esses resultados?

- **Tabela de elegibilidade:** mostra quantas unidades de análise chegam à Random Forest. Se a população elegível não for a mesma dos autores, o total de alertas pode divergir mesmo com modelo equivalente.
- **Gráfico A:** apresenta a distribuição dos *scores* (probabilidades estimadas pelo modelo). Concentração próxima de 0 indica poucos candidatos classificados como mixers no limiar 0,50.
- **Gráfico B:** mostra como mudar o limiar **mudaria** a quantidade de alertas. A linha de 63 é uma **referência do artigo, não uma meta de ajuste**.
- **Ranking:** permite auditar o endereço e a pontuação exata de cada candidato no nosso modelo. Sem a lista dos 63 autores, ainda não é possível conferir coincidência endereço a endereço com o artigo.

**Próxima hipótese científica:** conferir se a população, a transformação dos dados, a seleção dos 100 negativos e as regras de treino/inferência coincidem com o artigo. O e-mail aos autores sobre código, amostras e lista de 63 alertas já foi enviado.


In [ ]:
#@title 6. Salvar evidências do diagnóstico no seu Drive próprio
SALVAR_DIAGNOSTICO = True  #@param {type:"boolean"}
if SALVAR_DIAGNOSTICO:
    from googleapiclient.http import MediaFileUpload
    pasta_local = BASE / 'results' / 'diagnostico_63_vs_9'
    pasta_local.mkdir(parents=True, exist_ok=True)
    predicoes.to_csv(pasta_local / 'scores_elegiveis_2020.csv', index=False)
    curva.to_csv(pasta_local / 'curva_limiares.csv', index=False)
    contagens.to_csv(pasta_local / 'contagens.csv', index=False)
    fig.savefig(pasta_local / 'graficos_diagnostico.png', dpi=160, bbox_inches='tight')
    manifesto_diag = {
        'experimento': 'diagnostico_63_vs_9',
        'executado_em_utc': datetime.now(timezone.utc).isoformat(),
        'limiar_protocolo': 0.50, 'autores_alertas_reportados': 63,
        'treinamento': 'mesmo modelo ajustado na etapa 5 deste notebook',
        'min_tx': MIN_TX, 'seed': SEED,
        'hashes_dados': HASHES_ESPERADOS, 'hashes_codigo': EXPECTED_MODULE_SHA256,
        'observacao': 'diagnostico exploratorio, sem ajuste de hiperparametros para aproximar os 63 alertas',
    }
    (pasta_local / 'manifesto.json').write_text(json.dumps(manifesto_diag, indent=2, ensure_ascii=False))
    nome = 'diagnostico_63_vs_9_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    pasta_id = drive_api.files().create(body={
        'name': nome, 'mimeType': 'application/vnd.google-apps.folder',
        'parents': [PASTA_RESULTADOS_ID]}, fields='id').execute()['id']
    tipos = {'.csv': 'text/csv', '.json': 'application/json', '.png': 'image/png'}
    for arquivo in pasta_local.iterdir():
        if arquivo.suffix not in tipos:
            continue
        drive_api.files().create(body={'name': arquivo.name, 'parents': [pasta_id]},
             media_body=MediaFileUpload(str(arquivo), mimetype=tipos[arquivo.suffix]),
             fields='id').execute()
    print('Diagnóstico salvo:', 'https://drive.google.com/drive/folders/' + pasta_id)
else:
    print('Gráficos visualizados. Salvamento desativado.')


## 9 — Identificar três alertas ainda desconhecidos (consulta externa)

**Pergunta:** os três endereços não rotulados restantes correspondem a contratos conhecidos, contas comuns ou serviços que se parecem estatisticamente com mixers?

**Por quê:** um score alto **não prova** que um contrato é mixer. Primeiro precisamos coletar metadados do endereço e evidências verificáveis; depois examinaremos o comportamento **especificamente em 29/10/2020**. A consulta abaixo usa a API pública do [Blockscout](https://eth.blockscout.com/), em modo somente leitura; não exige token. **Pode haver bloqueio/rate limit**, e ausência de dados não é prova de que o contrato não existe.

**O que será observado:** se a API conseguiu responder, se o endereço é marcado como contrato, se há nome ou código verificado, e referências para inspeção. **Não transforma automaticamente metadados em rótulos negativos ou positivos.** Execute depois de entrar na sua conta Google na etapa 2.

In [ ]:
#@title 7. Consultar metadados públicos e guardar evidências (sem alterar rótulos)
import requests
from datetime import datetime, timezone
from googleapiclient.http import MediaFileUpload

ALVOS = {
    '0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9': 1.00,
    '0x98f1858c9740b9c5f2cd8abea67948873d38dc39': 0.98,
    '0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18': 0.76,
}

def consultar(url):
    try:
        resposta = requests.get(url, timeout=20, headers={'Accept':'application/json'})
        dados = resposta.json() if resposta.ok else {}
        return resposta.status_code, dados if isinstance(dados, dict) else {}, ''
    except (requests.RequestException, ValueError) as erro:
        return None, {}, type(erro).__name__ + ': ' + str(erro)[:160]

registros, respostas = [], {}
for endereco, score in ALVOS.items():
    base_url = f'https://eth.blockscout.com/api/v2/addresses/{endereco}'
    st, dados, erro = consultar(base_url)
    st_contrato, dados_contrato, erro_contrato = consultar(
        f'https://eth.blockscout.com/api/v2/smart-contracts/{endereco}')
    respostas[endereco] = {'url':base_url, 'http_endereco':st, 'dados_endereco':dados,
                            'http_contrato':st_contrato, 'dados_contrato':dados_contrato,
                            'erro':erro, 'erro_contrato':erro_contrato}
    registros.append({'endereco':endereco, 'score_nosso':score, 'http_endereco':st,
                      'is_contract':dados.get('is_contract'), 'nome':dados.get('name'),
                      'is_verified':dados.get('is_verified'),
                      'http_codigo':st_contrato, 'nome_codigo':dados_contrato.get('name'),
                      'fonte':base_url, 'erro':erro or erro_contrato})

metadados = pd.DataFrame(registros)
display(metadados)
print('Metadados são pistas, NÃO rótulos confirmados. Inspecionar atividade histórica de 2020 depois.')

SALVAR_CONSULTA = True  #@param {type:"boolean"}
if SALVAR_CONSULTA:
    pasta_local = BASE / 'results' / 'triagem_onchain_3'
    pasta_local.mkdir(parents=True, exist_ok=True)
    metadados.to_csv(pasta_local/'metadados_3_enderecos.csv', index=False)
    (pasta_local/'respostas_publicas.json').write_text(
        json.dumps({'data_utc':datetime.now(timezone.utc).isoformat(), 'respostas':respostas},
                   ensure_ascii=False, indent=2, default=str))
    nome = 'triagem_onchain_3_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    pasta_id = drive_api.files().create(body={
        'name':nome, 'mimeType':'application/vnd.google-apps.folder',
        'parents':[PASTA_RESULTADOS_ID]}, fields='id').execute()['id']
    for arquivo in (pasta_local/'metadados_3_enderecos.csv', pasta_local/'respostas_publicas.json'):
        media = MediaFileUpload(str(arquivo), mimetype='text/csv' if arquivo.suffix=='.csv' else 'application/json')
        drive_api.files().create(body={'name':arquivo.name, 'parents':[pasta_id]},
                                 media_body=media, fields='id').execute()
    print('Evidências salvas:', 'https://drive.google.com/drive/folders/' + pasta_id)


## 10 — Auditoria histórica: o que os três contratos fizeram em 29/10/2020?

**Pergunta:** as pontuações de 1,00, 0,98 e 0,76 podem ser conferidas a partir das transações reais daquele dia?

**Por que essa etapa é melhor do que consultar apenas o nome atual:** o nome no explorador não revela a atividade de 2020. Aqui usamos a **cópia do arquivo intermediário de transações de 29/10/2020** no seu Drive, filtramos apenas os três destinatários e recalculamos as quatro features com a função versionada do projeto. É uma **auditoria de consistência com os dados intermediários**, não uma confirmação independente das transações on-chain nem da identidade econômica dos protocolos.

**O que observar:** tabela de contagens; diferenças absolutas entre features recalculadas e processadas; gráfico que explica as quatro medidas. Para os três endereços, verifique que `n_tx` é respectivamente **53, 10 e 10** (conforme os relatórios anteriores). **Se qualquer comparação falhar, não interprete como conclusão: investigue esquema/fonte/transformação.**

**Observação sobre o proxy `0x5e60…`:** o JSON anterior aponta implementação inicial `0x68b37D89e1AEd3Dd348AD8B1cFC1E678539431D7` e implementação atual `0xee2fCEcFb2075A85a31eE8676cE9Ae5f3c3be6D7`. Não sabemos **a data da troca**; isso exige análise de eventos `Upgraded` e histórico de execução no bloco, fora do escopo deste check de features.

**Ação:** basta executar a próxima célula depois da etapa 2 (login no Google e módulos carregados). Ela produz os gráficos e pode gravar um novo relatório sem modificar os dados. O download intermediário tem cerca de 141 MB.

In [ ]:
#@title 8. Auditar transações de 2020 e comparar as 4 features
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from datetime import datetime, timezone
from googleapiclient.http import MediaFileUpload
import json, hashlib

SALVAR_AUDITORIA_HISTORICA = True  #@param {type:"boolean"}
ARQUIVO_INTERIM_2020 = '1Ei7sCgYdFfX-AjrK6uazOvbrf28d1Px7'
alvos = ['0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9',
         '0x98f1858c9740b9c5f2cd8abea67948873d38dc39',
         '0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18']
origem = obter_arquivo_drive(ARQUIVO_INTERIM_2020, BASE/'data'/'interim'/'test_20201029.parquet')
transacoes = pd.read_parquet(origem)
necessarias = {'recipient','hash','time','sender','call_count'}
assert necessarias.issubset(transacoes.columns), f'Colunas esperadas ausentes: {necessarias-set(transacoes.columns)}'
filtradas = transacoes.loc[transacoes['recipient'].astype(str).str.lower().isin(alvos)].copy()
assert not filtradas.empty, 'Os três destinatários não aparecem nesse arquivo intermediário.'
filtradas = preparar(filtradas)
assert filtradas['dia'].astype(str).eq('2020-10-29').all(), 'Há transações fora do dia de análise.'
recalculado = calcular_features(filtradas).drop(columns=['dia']).set_index('recipient')
ref = pd.read_parquet(INPUTS['teste2020']).set_index('recipient').loc[alvos]
comparacao = recalculado[['n_tx',*FEATURES]].join(ref[['n_tx',*FEATURES]], lsuffix='_novo', rsuffix='_referencia')
for metrica in ('n_tx',*FEATURES):
    comparacao[f'delta_{metrica}'] = (comparacao[f'{metrica}_novo']-comparacao[f'{metrica}_referencia']).abs()
comparacao['paridade_numerica'] = comparacao[[f'delta_{x}' for x in ('n_tx',*FEATURES)]].max(axis=1).le(1e-9)
resumo_tx = filtradas.groupby('recipient',as_index=False).agg(n_tx=('hash','size'),
                         remetentes_distintos=('sender','nunique'),
                         chamadas_medias=('call_count','mean'))
display(comparacao.round(6))
display(resumo_tx.round(2))
assert sorted(resumo_tx.n_tx.tolist()) == [10,10,53], 'Contagens inesperadas: investigar dados antes de concluir.'
fig_h, ax = plt.subplots(figsize=(9, 3.8))
view = comparacao[[f'delta_{f}' for f in FEATURES]].T
ax.imshow(np.log10(view.to_numpy()+1e-12), aspect='auto', vmin=-12, vmax=0, cmap='Blues')
ax.set_yticks(range(len(FEATURES)), list(FEATURES))
ax.set_xticks(range(len(alvos)), [x[:8]+'…'+x[-4:] for x in comparacao.index])
ax.set_title('Diferença absoluta das features recalculadas (quanto mais claro, menor)')
for i in range(len(FEATURES)):
    for j in range(len(alvos)):
        ax.text(j,i,f'{view.iloc[i,j]:.1e}',ha='center',va='center',fontsize=8)
fig_h.tight_layout(); plt.show()
print('Paridade numérica em todos os três:', bool(comparacao.paridade_numerica.all()))
print('AVISO: nomes atuais de proxies NÃO estabelecem sua função ou implementação em 2020.')
if SALVAR_AUDITORIA_HISTORICA:
    out=BASE/'results'/'auditoria_historica_2020'; out.mkdir(parents=True,exist_ok=True)
    comparacao.reset_index().to_csv(out/'comparacao_features_3.csv',index=False)
    resumo_tx.to_csv(out/'resumo_transacoes_3.csv',index=False)
    fig_h.savefig(out/'delta_features_3.png',dpi=150,bbox_inches='tight')
    (out/'manifesto.json').write_text(json.dumps({'data_utc':datetime.now(timezone.utc).isoformat(),
       'dataset_drive_id':ARQUIVO_INTERIM_2020,'sha256_interim':hashlib.sha256(origem.read_bytes()).hexdigest(),
       'origem':'interim 2020; auditoria de consistência, não auditoria independente da blockchain',
       'contratos':alvos,'paridade_por_contrato':comparacao['paridade_numerica'].to_dict()},indent=2,ensure_ascii=False))
    pasta=drive_api.files().create(body={'name':'auditoria_historica_2020_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'),
       'mimeType':'application/vnd.google-apps.folder','parents':['1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI']},fields='id').execute()['id']
    for arq in out.iterdir():
        if arq.suffix not in ('.csv','.png','.json'): continue
        mime={'.csv':'text/csv','.png':'image/png','.json':'application/json'}[arq.suffix]
        drive_api.files().create(body={'name':arq.name,'parents':[pasta]},media_body=MediaFileUpload(str(arq),mimetype=mime),fields='id').execute()
    print('Auditoria histórica salva:', 'https://drive.google.com/drive/folders/'+pasta)
